# 02 - Evaluation Scoring (Optimized)

**Features:**
- **Skip existing**: Only evaluates responses without evaluations
- **ThreadPoolExecutor**: Parallel evaluation calls
- **tqdm progress**: Visual progress bar
- **Batch inserts**: Accumulates records before inserting

In [8]:
import sys
sys.path.insert(0, '..')
sys.path.insert(0, '../..')

import pandas as pd
import json
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm
from sqlalchemy import text
from openai import OpenAI

from ares.db.connection import get_engine
from ares.db.operations import insert_evaluations
from ares.configs.db_config import TABLES, MODEL_NAMES
from ares.evaluation.evaluation import Scorer, GliderEvaluator, SemanticF1Evaluator

engine = get_engine()
print('Connected!')

Connected!


In [9]:
# Configuration
EVALUATOR_BASE_URL = 'http://localhost:8805/v1'
EVALUATOR_MODEL = 'PatronusAI/glider'
BATCH_SIZE = 50
MAX_SAMPLES = None  # Set to None for all
MAX_WORKERS = 8     # Parallel evaluation threads
RUN_SEMANTIC_F1 = False

evaluator_client = OpenAI(api_key='EMPTY', base_url=EVALUATOR_BASE_URL)

try:
    models = evaluator_client.models.list()
    print(f'Connected to: {[m.id for m in models.data]}')
except Exception as e:
    print(f'Warning: {e}')

Connected to: ['PatronusAI/glider']


In [10]:
# Chat functions for evaluators
def glider_chat_fn(messages, max_tokens=1024, model_name=None):
    try:
        response = evaluator_client.chat.completions.create(
            model=model_name or EVALUATOR_MODEL,
            messages=messages,
            max_tokens=max_tokens,
            temperature=0.0,
        )
        return response.choices[0].message.content
    except Exception as e:
        print(f'Error: {e}')
        return ''

def semantic_chat_fn(messages, max_tokens=1024):
    return glider_chat_fn(messages, max_tokens)

# Initialize evaluators
glider_evaluator = GliderEvaluator(chat_fn=glider_chat_fn, model_name=EVALUATOR_MODEL)
semantic_evaluator = SemanticF1Evaluator(chat_fn=semantic_chat_fn)
print('Evaluators ready!')

Evaluators ready!


In [11]:
# Load responses to evaluate (skip existing)
limit_clause = f'LIMIT {MAX_SAMPLES}' if MAX_SAMPLES else ''

query = f'''
SELECT r.sample_id, r.model_name, r.response_raw, r.response_id,
       s.prompt_text, s.ground_truth, s.ground_truth_type
FROM vlm_responses r
JOIN vlm_samples s ON r.sample_id = s.sample_id
LEFT JOIN vlm_evaluations e ON r.sample_id = e.sample_id AND r.model_name = e.model_name
WHERE e.evaluation_id IS NULL AND r.ok = true AND r.response_raw IS NOT NULL
ORDER BY r.sample_id, r.model_name
{limit_clause}
'''

df_to_eval = pd.read_sql(query, engine)
print(f'Responses to evaluate: {len(df_to_eval)}')
print(f'Unique samples: {df_to_eval["sample_id"].nunique()}')

Responses to evaluate: 98415
Unique samples: 24508


In [12]:
# Single evaluation function (for parallelization)
def evaluate_single_row(row):
    '''Evaluate a single response row.'''
    try:
        result = glider_evaluator.evaluate(
            question=row['prompt_text'],
            model_answer=row['response_raw'],
            ground_truth=row['ground_truth'],
            sample_id=row['sample_id'],
        )
        
        record = {
            'sample_id': row['sample_id'],
            'model_name': row['model_name'],
            'response_id': row.get('response_id'),
            'glider_score': result['score'],
            'glider_reasoning': result['reasoning'],
            'glider_highlight': json.dumps(result['highlight']) if result.get('highlight') else None,
            'glider_raw_output': result['raw_output'],
            'semantic_f1_precision': None, 'semantic_f1_recall': None,
            'semantic_f1_f1': None, 'semantic_f1_gen_statements': None,
            'semantic_f1_gt_statements': None, 'semantic_f1_matches': None,
            'semantic_f1_labels': None,
        }
        return ('success', record)
    except Exception as e:
        return ('error', str(e))

In [ ]:
# Run parallel evaluation with tqdm
print(f'Running parallel evaluation with {MAX_WORKERS} workers...')

evaluation_records = []
failed = 0
rows = df_to_eval.to_dict('records')

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(evaluate_single_row, row): i for i, row in enumerate(rows)}
    
    with tqdm(total=len(futures), desc='Evaluating') as pbar:
        for future in as_completed(futures):
            status, result = future.result()
            if status == 'success':
                evaluation_records.append(result)
            else:
                failed += 1
            
            # Batch insert
            if len(evaluation_records) >= BATCH_SIZE:
                insert_evaluations(evaluation_records, engine)
                pbar.set_postfix({'inserted': len(evaluation_records)})
                evaluation_records = []
            
            pbar.update(1)

# Final batch
if evaluation_records:
    insert_evaluations(evaluation_records, engine)
    print(f'Final batch: {len(evaluation_records)}')

print(f'\nDone! Failed: {failed}')

Running parallel evaluation with 8 workers...


Evaluating:   0%|          | 0/98415 [00:00<?, ?it/s]

In [ ]:
# Verify
query = '''SELECT model_name, COUNT(*) as count, ROUND(AVG(glider_score)::numeric, 2) as avg_score
FROM vlm_evaluations GROUP BY model_name ORDER BY avg_score DESC'''
pd.read_sql(query, engine)

---
## Optional: Semantic F1 Evaluation

In [ ]:
# Run Semantic F1 (if enabled)
if RUN_SEMANTIC_F1:
    print('Running Semantic F1 evaluation...')
    
    # Get rows without semantic F1
    query = '''SELECT e.evaluation_id, e.sample_id, e.model_name,
                      r.response_raw, s.ground_truth
               FROM vlm_evaluations e
               JOIN vlm_responses r ON e.sample_id = r.sample_id AND e.model_name = r.model_name
               JOIN vlm_samples s ON e.sample_id = s.sample_id
               WHERE e.semantic_f1_f1 IS NULL
               LIMIT 100'''
    df_sem = pd.read_sql(query, engine)
    
    for idx, row in tqdm(df_sem.iterrows(), total=len(df_sem), desc='Semantic F1'):
        try:
            result = semantic_evaluator.evaluate(
                ground_truth=row['ground_truth'],
                response=row['response_raw']
            )
            
            update = text('''
                UPDATE vlm_evaluations SET
                    semantic_f1_precision = :precision,
                    semantic_f1_recall = :recall,
                    semantic_f1_f1 = :f1,
                    updated_at = NOW()
                WHERE evaluation_id = :eval_id
            ''')
            
            with engine.begin() as conn:
                conn.execute(update, {
                    'precision': result['precision'],
                    'recall': result['recall'],
                    'f1': result['f1'],
                    'eval_id': row['evaluation_id']
                })
        except Exception as e:
            print(f'Error: {e}')
    
    print('Semantic F1 done!')
else:
    print('Semantic F1 skipped. Set RUN_SEMANTIC_F1 = True to run.')

In [ ]:
# Summary
with engine.connect() as conn:
    total = conn.execute(text('SELECT COUNT(*) FROM vlm_evaluations')).scalar()
    avg_glider = conn.execute(text('SELECT ROUND(AVG(glider_score)::numeric, 2) FROM vlm_evaluations')).scalar()
    
print(f'Total evaluations: {total}')
print(f'Average Glider score: {avg_glider}/5')